In [ ]:
!pip install -q -U sentence-transformer chromadb langchain-text-splitters py pdf

ERROR: Could not find a version that satisfies the requirement sentence-transformer (from versions: none)
ERROR: No matching distribution found for sentence-transformer


In [ ]:
!pip install chromadb


In [ ]:
!pip install langchain-text-splitters



In [ ]:
!pip install pypdf

In [ ]:
import os
from google.colab import userdata, files
import requests
from sentence_transformers import SentenceTransformer
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader


In [ ]:
api_key = os.environ.get("OPENROUTER_API_KEY")
if not api_key:
  try:
    api_key = userdata.get("langchain_genaicourse")
  except Exception:
    pass
  os.environ["OPENROUTER_API_KEY"] = api_key


In [ ]:
print(" Please upload one or more PDF files:")
uploaded = files.upload()

pdf_texts = []
for filename in uploaded.keys():
  if filename.endswith('.pdf'):
    reader = PdfReader(filename)
    text = ""
    for page_num, page in enumerate(reader.pages):
      page_text = page.extract_text()
      if page_text:
        text += f"\n --- Page {page_num +1} ---\n" + page_text
    pdf_texts.append(text)
    print(f" Loaded '{filename}' ({len(reader.pages)} pages).")

if not pdf_texts:
  raise ValueError("No valid PDF files uploaded. Please re-run and upload a .pdf")

full_pdf_content = "\n\n".join(pdf_texts)

 Please upload one or more PDF files:


Saving BDAV (1,2).pdf to BDAV (1,2).pdf
 Loaded 'BDAV (1,2).pdf' (79 pages).


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
chunks = text_splitter.split_text(full_pdf_content)
print(f" Extracted and split document into {len(chunks)} text chunks.")


 Extracted and split document into 201 text chunks.


In [ ]:
print(" Loading embedding model and building vector index...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.Client()

#Reset collection for clean execution
try:
  chroma_client.delete_collection(name="pdf_rag_collection")
except Exception:
  pass

collection = chroma_client.create_collection(name="pdf_rag_collection")

#Embed chunks in batches
chunk_embeddings = embedder.encode(chunks).tolist()
chunks_ids = [f"doc_chunks_{i}" for i in range(len(chunks))]

collection.add(
    documents=chunks,
    embeddings=chunk_embeddings,
    ids=chunks_ids
)

print("PDF Vector Indexing Complete\n")

 Loading embedding model and building vector index...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

PDF Vector Indexing Complete



In [ ]:
def retrieve_pdf_context(query: str, top_k:int=3) -> list[str]:
  query_embedding = embedder.encode(query).tolist()
  results = collection.query(
      query_embeddings=query_embedding,
      n_results=top_k
  )
  return results["documents"][0]

def ask_pdf(query: str):
  context_passages = retrieve_pdf_context(query, top_k=3)
  context_str = "\n".join(f"- {p}" for p in context_passages)

  prompt = f"""You are an intelligent document analysis assistant. Answer the question using ONLY the provided information.If the information is not contained within the provided context, state clearly: 'I cannot find the answer in the provided PDF'"

PDF Context:
{context_str}

Question: {query}
Answer:"""

  response = requests.post(
      url="https://openrouter.ai/api/v1/chat/completions",
      headers={
          "Authorization": f"Bearer {api_key}",
          "Content-Type": "application/json"
      },
      json={
          "model": "openrouter/free",
          "messages": [{"role": "user", "content": prompt}]
      }
  )

  return response.json()["choices"][0]["message"]["content"], context_passages

In [ ]:
print("=" * 60)
print(" PDF CHATBOT READY! Type your question below (or type 'exit' to quit).")
print("=" * 60)

while True:
  user_query = input("\nAsk a question about your PDF: ")
  if user_query.lower() in ["exit","quit","q"]:
    print(" Exiting PDF Chatbot. Goodbye!")
    break
  if not user_query.strip():
    continue

  answer, context = ask_pdf(user_query)

  print("\n--- RETRIEVED PDF SNIPPETS ---")
  for i,snippet in enumerate(context, 1):
    print(f"[{i}] {snippet[:150]}...")

  print("\n--- GEMINI RESPONSE ---")
  print(answer)
  print("=" * 60)


 PDF CHATBOT READY! Type your question below (or type 'exit' to quit).

Ask a question about your PDF: What is the difference between NameNode and DataNode?

--- RETRIEVED PDF SNIPPETS ---
[1] NameNode
 
=
 
Master
 
+
 
Metadata
 
management
 
2.
 
DataNode
 
The
 
DataNode
 
is
 
the
 
worker/storage
 
node
 
of
 
HDFS.
 
Its
 
main
 
func...
[2] NameNode
 
is
 
the
 
master
 
node
 
of
 
HDFS.
 
Its
 
main
 
functions
 
are:
 
●
 
Maintains
 
information
 
about
 
files
 
and
 
directories.
 
...
[3] In
 
simple
 
words:
 
DataNode
 
=
 
Actual
 
Data
 
Storage
 
3.
 
Secondary
 
NameNode
 
The
 
Secondary
 
NameNode
 
helps
 
the
 
NameNode
 
by
 ...

--- GEMINI RESPONSE ---
User Safety: safe

Ask a question about your PDF: What are the 5 V's of Big Data?

--- RETRIEVED PDF SNIPPETS ---
[1] Data
 
requires
 
distributed
 
storage,
 
parallel
 
processing
 
and
 
faster
 
analytics
.
 
 
2.
 
Characteristics
 
of
 
Big
 
Data
 
—
 
5
 
V's...
[2] —
 
5
 
V's
 
of
 
Big
 
Data
 
    